# Atividade de Processamento de Imagens

**Aluno:** _Vinícius_ &nbsp;|&nbsp; **Prof.:** Matheus Araújo &nbsp;|&nbsp; **Tema do TF:** Eleição de comandante em frota de drones

---

## Objetivo

Explorar e consolidar conceitos de **compressão**, **representação** e **descrição** de imagens
digitais por meio da implementação prática de duas técnicas fundamentais:

1. **Compressão por DCT** (inspirada no JPEG) — divisão em blocos, transformada, quantização e reconstrução.
2. **Descritores estatísticos e estruturais** — média, variância, energia e variação espacial.

> **Regras atendidas:** as bibliotecas OpenCV/PIL são usadas **apenas para carregar e salvar** imagens.
> Toda transformação (DCT, IDCT, quantização e os descritores) é implementada **manualmente**.
> O `numpy` é usado somente como aritmética geral (somas e produtos de matrizes) e o `matplotlib`
> apenas para exibir os resultados no relatório.


## 1. Configuração e carregamento de imagens

Coloque aqui os caminhos das **suas imagens de drone** (tema do trabalho final).
Sugestão para a Questão 2: uma imagem com **muito detalhe estrutural** (área urbana/vegetação vista de cima)
e outra **homogênea** (céu, mar, plantação uniforme).

Se os arquivos não forem encontrados, o notebook gera imagens **sintéticas de demonstração**
(claramente marcadas) só para rodar de ponta a ponta — **substitua pelas suas imagens reais antes de entregar.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image          # PIL: APENAS carregar/salvar imagens
import os

# >>> EDITE AQUI com os caminhos das suas imagens de drone <<<
IMG_Q1 = "imagens/drone_cena.jpg"      # imagem para a compressão (Questão 1)
IMG_A  = "imagens/drone_textura.jpg"   # imagem com muito detalhe (Questão 2)
IMG_B  = "imagens/drone_ceu.jpg"       # imagem homogênea          (Questão 2)


In [ ]:
def carregar(caminho):
    """Carrega a imagem como matriz numpy. (PIL usado SÓ para abrir o arquivo.)"""
    return np.asarray(Image.open(caminho), dtype=np.float64)

def para_cinza(arr):
    """Conversão RGB->tons de cinza implementada MANUALMENTE (luminância Rec. 601).
       Y = 0.299 R + 0.587 G + 0.114 B."""
    if arr.ndim == 2:                       # já é cinza
        return arr.copy()
    a = arr[..., :3]                        # ignora canal alfa se existir
    return 0.299*a[..., 0] + 0.587*a[..., 1] + 0.114*a[..., 2]

def salvar_cinza(arr, caminho):
    """Salva matriz como imagem (PIL usado SÓ para salvar)."""
    Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8), mode="L").save(caminho)

# ---- gerador de imagem sintética (fallback de demonstração) -------------
def _sintetica(tipo, M=256, N=256):
    yy, xx = np.mgrid[0:M, 0:N]
    if tipo == "textura":      # alta variação local (simula vegetação/cidade)
        img = 110 + 55*np.sin(xx/3.0)*np.cos(yy/3.0) + 25*np.sin(xx/1.3 + yy/1.7)
    elif tipo == "ceu":        # baixa variação (simula céu)
        img = 60 + 0.25*yy + 8*np.sin(xx/30.0)
    else:                      # cena mista p/ Q1
        ceu = 60 + 0.25*yy + 8*np.sin(xx/30.0)
        veg = 110 + 55*np.sin(xx/3.0)*np.cos(yy/3.0)
        img = np.where(yy < M*0.45, ceu, veg)
    return np.clip(img, 0, 255)

def carregar_cinza(caminho, tipo_fallback):
    if os.path.exists(caminho):
        print(f"[OK] carregada: {caminho}")
        return para_cinza(carregar(caminho)), False
    print(f"[AVISO] '{caminho}' nao encontrado -> usando imagem SINTETICA de demonstracao. "
          f"Substitua pela sua imagem de drone antes de entregar!")
    return _sintetica(tipo_fallback), True

img_q1, _ = carregar_cinza(IMG_Q1, "mista")
img_a,  _ = carregar_cinza(IMG_A,  "textura")
img_b,  _ = carregar_cinza(IMG_B,  "ceu")
print("dimensoes:", img_q1.shape, img_a.shape, img_b.shape)

## 2. Questão 1 — Compressão por DCT (estilo JPEG)

### 2.1 Ideia

O JPEG explora o fato de que, em blocos pequenos, a maior parte da energia da imagem se concentra
em **poucas frequências baixas**. O processo é:

$$\text{cinza} \;\to\; \text{blocos } 8\times8 \;\to\; \text{DCT 2D} \;\to\; \text{quantização} \;\to\;
(\text{dequantização}) \;\to\; \text{IDCT} \;\to\; \text{recompor}$$

A **DCT-II 2D** de um bloco $f$ é:

$$F(u,v)=\frac{2}{N}\,C(u)C(v)\sum_{x=0}^{N-1}\sum_{y=0}^{N-1} f(x,y)\,
\cos\!\Big[\tfrac{(2x+1)u\pi}{2N}\Big]\cos\!\Big[\tfrac{(2y+1)v\pi}{2N}\Big]$$

com $C(0)=1/\sqrt2$ e $C(k)=1$ para $k>0$.

Implementamos a DCT de forma **separável** construindo a matriz ortonormal $T$ a partir da definição
do cosseno: $\;F = T\,B\,T^{\top}\;$ e $\;B = T^{\top} F\,T\;$ (como $T$ é ortonormal, a inversa é a transposta).
A **perda** acontece só na quantização: $F_q=\mathrm{round}(F/Q)$.

In [ ]:
def matriz_dct(N):
    """Matriz de transformacao DCT-II ortonormal NxN, construida da definicao:
       T[u,x] = a(u) * cos((2x+1) u pi / 2N),  a(0)=sqrt(1/N), a(u>0)=sqrt(2/N)."""
    T = np.zeros((N, N))
    for u in range(N):
        cu = np.sqrt(1.0/N) if u == 0 else np.sqrt(2.0/N)
        for x in range(N):
            T[u, x] = cu * np.cos((2*x + 1) * u * np.pi / (2.0*N))
    return T

def dct2d(bloco, T):
    return T @ bloco @ T.T        # DCT 2D separavel

def idct2d(coef, T):
    return T.T @ coef @ T         # IDCT 2D separavel

In [ ]:
# Matriz de quantizacao de luminancia padrao JPEG
Q_LUM = np.array([
    [16,11,10,16,24,40,51,61],
    [12,12,14,19,26,58,60,55],
    [14,13,16,24,40,57,69,56],
    [14,17,22,29,51,87,80,62],
    [18,22,37,56,68,109,103,77],
    [24,35,55,64,81,104,113,92],
    [49,64,78,87,103,121,120,101],
    [72,92,95,98,112,100,103,99]], dtype=np.float64)

def escalar_quantizacao(Q, qualidade):
    """Escala Q por um fator de qualidade (1-100). Qualidade baixa => passos
       maiores => mais compressao e mais artefatos (mesma regra do JPEG)."""
    qualidade = max(1, min(100, qualidade))
    s = 5000.0/qualidade if qualidade < 50 else 200.0 - 2.0*qualidade
    Qs = np.floor((Q*s + 50.0)/100.0)
    Qs[Qs == 0] = 1.0
    return Qs

In [ ]:
def comprimir_descomprimir(cinza, Q, N=8):
    """Pipeline bloco-a-bloco: level shift -> DCT -> quantiza -> dequantiza -> IDCT.
       Retorna (imagem_reconstruida, percentual_de_coeficientes_nao_nulos)."""
    M, C = cinza.shape
    pb, pr = (-M) % N, (-C) % N                       # padding p/ multiplo de N
    img = np.pad(cinza, ((0, pb), (0, pr)), mode="edge")
    T = matriz_dct(N)
    recon = np.zeros_like(img, dtype=np.float64)
    nz = tot = 0
    for i in range(0, img.shape[0], N):
        for j in range(0, img.shape[1], N):
            bloco  = img[i:i+N, j:j+N] - 128.0         # level shift
            coef   = dct2d(bloco, T)
            coef_q = np.round(coef / Q)                # <- QUANTIZACAO (perda)
            nz += np.count_nonzero(coef_q); tot += coef_q.size
            recon[i:i+N, j:j+N] = idct2d(coef_q * Q, T) + 128.0
    return np.clip(recon[:M, :C], 0, 255), 100.0*nz/tot

def mse(a, b):  return float(np.mean((a.astype(float) - b.astype(float))**2))
def psnr(a, b):
    e = mse(a, b)
    return float("inf") if e == 0 else float(10*np.log10(255.0**2 / e))

### 2.2 Execução e comparação para diferentes níveis de qualidade

Quanto **menor** a qualidade, mais agressiva a quantização: o PSNR cai, sobram menos coeficientes
não nulos (maior compressão) e os **artefatos de bloco 8×8** ficam mais visíveis.

In [ ]:
qualidades = [90, 50, 10]
resultados = {}
for q in qualidades:
    rec, pct = comprimir_descomprimir(img_q1, escalar_quantizacao(Q_LUM, q))
    resultados[q] = rec
    print(f"qualidade={q:3d} | PSNR={psnr(img_q1, rec):6.2f} dB | "
          f"MSE={mse(img_q1, rec):8.2f} | coef. nao-nulos={pct:5.1f}%")

In [ ]:
fig, ax = plt.subplots(1, len(qualidades)+1, figsize=(4*(len(qualidades)+1), 4))
ax[0].imshow(img_q1, cmap="gray", vmin=0, vmax=255); ax[0].set_title("Original"); ax[0].axis("off")
for k, q in enumerate(qualidades, start=1):
    ax[k].imshow(resultados[q], cmap="gray", vmin=0, vmax=255)
    ax[k].set_title(f"q={q}  (PSNR {psnr(img_q1, resultados[q]):.1f} dB)"); ax[k].axis("off")
plt.tight_layout(); plt.show()

### 2.3 Onde o erro se concentra (mapa de diferença e artefatos de bloco)

In [ ]:
q = 10
rec = resultados[q]
dif = np.abs(img_q1 - rec)
fig, ax = plt.subplots(1, 3, figsize=(13, 4.2))
ax[0].imshow(img_q1, cmap="gray", vmin=0, vmax=255); ax[0].set_title("Original"); ax[0].axis("off")
ax[1].imshow(rec, cmap="gray", vmin=0, vmax=255);   ax[1].set_title(f"Reconstruida q={q}"); ax[1].axis("off")
m = ax[2].imshow(dif, cmap="inferno"); ax[2].set_title("|Diferenca|"); ax[2].axis("off")
fig.colorbar(m, ax=ax[2], fraction=0.046); plt.tight_layout(); plt.show()

# recorte para evidenciar os blocos 8x8 numa regiao lisa
y0, x0 = 0, 0
crop_o = img_q1[y0:y0+64, x0:x0+64]; crop_r = rec[y0:y0+64, x0:x0+64]
fig, ax = plt.subplots(1, 2, figsize=(7, 3.6))
ax[0].imshow(crop_o, cmap="gray", vmin=0, vmax=255); ax[0].set_title("Recorte original"); ax[0].axis("off")
ax[1].imshow(crop_r, cmap="gray", vmin=0, vmax=255); ax[1].set_title("Recorte reconstruido (blocos 8x8)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

salvar_cinza(rec, "reconstruida_q10.png")   # exemplo de salvamento (PIL)

### 2.4 Análise da Questão 1

> _Escreva aqui a sua análise com base nos números acima. Pontos esperados:_
>
> - **Trade-off qualidade × compressão:** à medida que a qualidade cai (90 → 50 → 10), o **PSNR diminui**
>   e o **percentual de coeficientes não nulos** também cai — ou seja, descartamos cada vez mais informação
>   de alta frequência e ganhamos compressão ao custo de qualidade.
> - **Perda de detalhe:** as regiões de **alta frequência** (vegetação/bordas, vistas do drone) são as primeiras
>   a degradar, porque a quantização zera justamente os coeficientes de alta frequência. Isso aparece como
>   borramento de textura e é visível no **mapa de diferença** (erro concentrado nessas áreas).
> - **Artefatos de bloco:** em qualidade baixa surge o típico **"blocking" 8×8** — descontinuidades nas bordas
>   dos blocos, mais perceptíveis em regiões lisas (como o céu), pois cada bloco é quantizado de forma independente.
> - Relacione com o seu TF: imagens aéreas de drone com muita textura (telhados, copas de árvores) sofrem mais
>   perda perceptível do que regiões homogêneas.

## 3. Questão 2 — Descritores estatísticos e estruturais

Para uma imagem em tons de cinza $f$ de tamanho $M\times N$ (com $n=MN$ pixels):

| Descritor | Fórmula | O que mede |
|---|---|---|
| Média | $\mu=\frac1n\sum f(x,y)$ | brilho médio |
| Variância | $\sigma^2=\frac1n\sum (f-\mu)^2$ | dispersão / contraste global |
| Energia | $E=\sum f(x,y)^2$ | intensidade total ao quadrado |
| Var. espacial (H) | $\frac{1}{M(N-1)}\sum|f(x,y{+}1)-f(x,y)|$ | mudança entre vizinhos horizontais |
| Var. espacial (V) | $\frac{1}{(M-1)N}\sum|f(x{+}1,y)-f(x,y)|$ | mudança entre vizinhos verticais |

A **variância** é um descritor global (não vê posição), enquanto a **variação espacial** captura a
*estrutura local* — é alta em texturas e bordas, e baixa em regiões homogêneas.

In [ ]:
def descritores(cinza):
    """Calcula descritores globais e estruturais MANUALMENTE."""
    f = cinza.astype(np.float64)
    M, N = f.shape; n = M*N
    media     = f.sum() / n
    variancia = ((f - media)**2).sum() / n
    energia_t = (f**2).sum()
    # variacao espacial: |diferenca| entre vizinhos
    dh = np.abs(f[:, 1:] - f[:, :-1])      # horizontal
    dv = np.abs(f[1:, :] - f[:-1, :])      # vertical
    return {
        "media":            media,
        "variancia":        variancia,
        "desvio_padrao":    np.sqrt(variancia),
        "energia_total":    energia_t,
        "energia_media":    energia_t / n,
        "var_espacial_h":   dh.sum() / dh.size,
        "var_espacial_v":   dv.sum() / dv.size,
        "var_espacial_tot": (dh.sum() + dv.sum()) / (dh.size + dv.size),
    }

### 3.1 Comparação entre duas imagens (texturizada × homogênea)

In [ ]:
da = descritores(img_a)   # esperada: mais detalhe (textura)
db = descritores(img_b)   # esperada: homogenea (ceu)

print(f"{'descritor':<18}{'IMG_A (textura)':>20}{'IMG_B (homogenea)':>20}")
print("-"*58)
for k in da:
    print(f"{k:<18}{da[k]:>20.3f}{db[k]:>20.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img_a, cmap="gray", vmin=0, vmax=255); ax[0].set_title("IMG_A (textura)"); ax[0].axis("off")
ax[1].imshow(img_b, cmap="gray", vmin=0, vmax=255); ax[1].set_title("IMG_B (homogenea)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

labels = ["variancia", "var.espacial", "energia media/100"]
va = [da["variancia"], da["var_espacial_tot"], da["energia_media"]/100]
vb = [db["variancia"], db["var_espacial_tot"], db["energia_media"]/100]
x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(x - w/2, va, w, label="IMG_A (textura)")
ax.bar(x + w/2, vb, w, label="IMG_B (homogenea)")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("valor"); ax.legend()
ax.set_title("Comparacao de descritores"); plt.tight_layout(); plt.show()

### 3.2 Análise da Questão 2

> _Interprete a tabela e o gráfico. Pontos esperados:_
>
> - **Variação espacial** é o descritor mais discriminativo: a imagem **texturizada** (vegetação/área urbana
>   do drone) tem variação espacial muito **maior** que a **homogênea** (céu), porque há muitas mudanças
>   bruscas de intensidade entre pixels vizinhos.
> - A **variância** também tende a ser maior na imagem detalhada, mas pode confundir: uma imagem homogênea
>   porém com gradiente suave de brilho pode ter variância alta e, ainda assim, variação espacial baixa —
>   por isso os dois descritores **se complementam** (global × local).
> - A **energia** depende fortemente do brilho médio; sozinha não distingue textura, mas ajuda a caracterizar
>   o nível geral de intensidade.
> - **Aplicação em classificação:** esse vetor de descritores `[média, variância, energia, var_H, var_V]`
>   pode alimentar um classificador simples para separar, por exemplo, *céu/água* (baixa var. espacial) de
>   *vegetação/edificações* (alta var. espacial) em imagens capturadas pela frota de drones — útil para
>   segmentação de cenário ou priorização de áreas de interesse.

## 4. Conclusão

- A **compressão por DCT** mostra na prática o compromisso entre taxa de compressão e qualidade: a perda é
  controlada pela matriz de quantização e se manifesta como perda de detalhe em altas frequências e
  artefatos de bloco 8×8 em baixa qualidade.
- Os **descritores** resumem numericamente o conteúdo visual; em especial, a **variação espacial** separa bem
  regiões texturizadas de homogêneas, o que é diretamente aplicável à análise automática de imagens aéreas
  no contexto do trabalho final (frota de drones).

> **Checklist de entrega:** trocar as imagens sintéticas pelas suas imagens reais de drone, conferir que os
> avisos `[AVISO]` sumiram (todas `[OK]`), rodar o notebook do início ao fim e exportar para PDF
> (no Colab: *Arquivo → Imprimir → Salvar como PDF*).